# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the **FAIR^2 Mass Spectrometry EV** dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema (JSON-LD) at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

> This notebook demonstrates reproducible FAIR data access and exploratory analysis, referencing all entities (`recordSet`, `field`, etc.) by their Croissant `@id` attribute.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant FAIR2 dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant schema for the FAIR2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review all available record sets, their `@id`s, and their field IDs to understand the schema structure.

In [ ]:
# List all record sets available in the dataset
print("Available Record Sets:\n---------------------")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set.name} (@id={record_set.id})")

print("\nExample: For each record set, show field IDs and their descriptions:\n")
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet: {record_set.name} (@id={record_set.id})")
    for field in record_set.fields:
        print(f"  - Field: {field.name}, @id: {field.id}, type: {field.data_type}")
    print()

## 3. Data Extraction
Load selected record sets as DataFrames for further exploration. All entities are referenced by their Croissant `@id`.

In [ ]:
# Gather all record set IDs (@id values)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set IDs available:", record_set_ids)

# Load all record sets into DataFrames, indexed by their @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display available columns for each record set
for rs_id, df in dataframes.items():
    print(f"\nRecord Set {rs_id} columns: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Let's select a main data table for analysis. We'll:
- Filter protein entries with high peptide counts
- Normalize molecular weight (MW)
- Group by protein function/description

> We reference columns by their Croissant `@id`, following best FAIR practices.

In [ ]:
# Pick primary protein data record set (edit this @id based on Section 2 output)
# Example: Suppose the main table @id is 'https://api.app.sen.science/frontiers/7154140/dataset', update as needed
main_rs_id = record_set_ids[0]  # change this to target your core record set
df = dataframes[main_rs_id]

# List all available field ids (columns) in this table
print("Fields in main record set:", df.columns)

# Choose a numeric field from the table: example: 'peptide_count' or similar (must exist in the table)
# To keep FAIR, ensure to use the column's @id as in the schema
# For demo, let's use 'cr:field/numberOfPeptides' as a plausible field id
possible_numeric_fields = [c for c in df.columns if 'peptide' in c.lower() or 'count' in c.lower() or 'mw' in c.lower()]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.select_dtypes('number').columns[0]
print("Selected numeric field for filtering:", numeric_field)

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold} (showing top 5):")
print(filtered_df.head())

# Normalize this field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records (top 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a text-based categorical field (e.g. 'description', or similar)
group_candidates = [c for c in df.columns if any(k in c.lower() for k in ["description", "function", "gene", "type"])]
group_field = group_candidates[0] if group_candidates else df.columns[0]

print(f"\nGrouping by: {group_field}")
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print("Grouped means (top 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize key properties of the record set.

- Distribution of the numeric field
- Top categories by mean value


In [ ]:
import matplotlib.pyplot as plt

# Histogram for the selected numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Top group field means
if 'grouped_df' in locals():
    topn = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(9,4))
    plt.barh(topn[group_field], topn[numeric_field])
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We loaded the FAIR2 mass spectrometry dataset using `mlcroissant` referencing all components by their Croissant `@id`.
- We provided schema and field introspection and basic EDA: filtering by protein peptide count, normalizing MW, and analyzing by protein type.
- This notebook can be reused for any Croissant schema by simply updating the URL and referencing field/table `@id`s.
